# Project imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

import random
from datetime import datetime
import time

from fontTools.merge.util import current_time

C# initialize flappy game

In [2]:
import sys
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'
sys.path.insert(0, 'itml-project2')
# noinspection PyUnresolvedReferences
from ple.games.flappybird import FlappyBird
# noinspection PyUnresolvedReferences
from ple import PLE

pygame 2.6.1 (SDL 2.28.4, Python 3.13.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
couldn't import doomish


In [3]:
class PPO_Flappy(nn.Module):

    def __init__(self,num_layers,layer_specs):
        super().__init__()

        self.layers = nn.ModuleList()
        for i in range(num_layers):
                self.layers.append(nn.Linear(layer_specs[i][0],layer_specs[i][1]))


        self.actor = nn.Linear(layer_specs[-1][1],2) # actor outputs a probability of jumping or doing nothing
        self.critic = nn.Linear(layer_specs[-1][1],1) # critic outputs value for state


    def forward(self,x):

        for i in self.layers:
            x = F.tanh(i(x))

        action_prob = F.softmax(self.actor(x),dim=-1) # return action probability
        state_value = self.critic(x) # Critic assessed state value
        return action_prob,state_value




# Helper Functions

In [4]:
def normalize_game_state(state):
    means = torch.tensor([256.0, 0.0, 200.0, 200.0, 200.0, 400.0, 200.0, 200.0])
    stds = torch.tensor([128.0, 5.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0])
    return (state - means) / stds

def create_network():
    total_layers = 3
    layer_specs = [[8, 1024], [1024, 512], [512, 2048]]
    #layer_specs = [[8, 1024], [1024, 128], [128, 16], [16, 256], [256, 256]]
    return total_layers,layer_specs

def get_children():
    total_layers = 5
    input_layer = 8
    layer_specs = []
    layer_setup = [[8, 512], [512, 128], [128, 8], [8, 256], [256, 512]]

    layer_specs.append([input_layer,int(layer_setup[0][1]*2**(random.randint(-1,1)))])
    for i in range(total_layers-1):
        layer_specs.append([layer_specs[i][1],int(layer_setup[i+1][1]*2**(random.randint(-1,1)))])
    return total_layers,layer_specs

def get_random_layers():
    input_layer = 8
    total_layer_count = 5
    total_layers = random.randint(1, total_layer_count)
    layer_specs = []
    layer_specs.append([input_layer, 2 ** (3 + random.randint(1, 8))])

    for i in range(total_layers - 1):
        layer_specs.append([layer_specs[i][1], int(layer_specs[i][1]*2 ** (random.randint(-2, 2)))])

    total_size = 0
    for i in layer_specs:
        size = i[0]*i[1] + i[1]
        total_size += size

    print(f"Total Network Size: {total_size}")

    return total_layers, layer_specs

def get_Advantage(states,rewards):
    # Calculating Advantage
    # Advantage = G - V(S_t),

    # V(S_t)
    with torch.no_grad():
        _,prev_values = model(states)

    # G calculations
    G = 0
    returns = [] # Real reward returns discounted
    for reward in reversed(rewards):
        G = reward + gamma*G
        G = torch.FloatTensor([G])
        returns.insert(0,G)
    returns = torch.stack(returns)

    A = (returns - prev_values).squeeze()

    return A, returns

def train_print(epoch_num,L_clip_mean,L_vf_mean,L_entr_mean,avg_reward,elapsed_time):
    print(f"Epoch: {epoch_num}, "
              f"L_clip: {L_clip_mean:.2f}, "
              f"Squared loss: {L_vf_mean:.2f}, "
              f"Entropy loss: {L_entr_mean:.2f}, "
              f"Avg Rewards: {avg_reward:.2f}, "
              f"Time elapsed: {elapsed_time:.2f}s")

def get_model_names(total_models):
    """ Generate Model names"""
    identifier = [i for i in range(total_models)]

    unique_str = datetime.now().strftime("%d-%m-%Y ")

    names = ["flappy weights/Flappy " + str(unique_str)+str(i) + ".pt" for i in identifier]
    return names

# training Loop

In [5]:
def Train_Network(epochs,print_freq):

    start_time = time.time()
    game = FlappyBird()

    total_rewards = []
    all_L_clip = []
    all_L_vf = []
    all_L_entropy = []
    stats_rows = []

    for i in range(epochs):
        done = False
        p = PLE(game, fps=30, display_screen=False, force_fps=True)
        p.init()

        log_probs = []
        rewards = []
        states = []
        actions = []

        while not done:
            game_state = normalize_game_state(torch.tensor(list(p.getGameState().values())))

            action_prob,_ = model(game_state) # get action prob from model, critic values called later

            # Actor Action
            action_prob = action_prob.detach().squeeze()
            dist = torch.distributions.Categorical(action_prob)
            action = dist.sample()

            if action == 0:
                ret_action = p.getActionSet()[0]
            else:
                ret_action = p.getActionSet()[1]

            # Acting on environment
            reward = p.act(ret_action)

            # Saving values
            log_probs.append(dist.log_prob(action))
            rewards.append(reward)
            actions.append(action)
            states.append(game_state)

            #print_state(action,state,reward)
            if p.game_over():
                done = True
                p.reset_game()


        # Storing values after game loop
        total_rewards.append(sum(rewards))
        states = torch.stack(states).squeeze()
        log_probs = torch.stack(log_probs).squeeze()
        actions = torch.stack(actions).squeeze()
        # Calculating advantage and storing returns for later
        A, returns = get_Advantage(states,rewards)


        for j in range(K_epochs):

            action_probs,new_values = model(states) # Generate new actions and values
            new_dist = torch.distributions.Categorical(action_probs) # Create distribution from new actions probs
            new_policy = new_dist.log_prob(actions) # New policy probs of prev actions

            ratio = torch.exp(new_policy-log_probs) # Ratio of new vs prev policy
            clipped_ratio = torch.clamp(ratio,min = 1-epsilon,max= 1+epsilon) # Ratio most stay between 1 +- eps

            L_clip = (torch.min(ratio*A,clipped_ratio*A).mean())*c0 # We select the ratio if between clip range else clipped ratio
            L_vf = ((new_values.squeeze() - returns.squeeze())**2)*c1 # Calculate squared loss
            entropy_bonus = (new_dist.entropy())*c2 # Getting entropy bonus for loss

            loss = -(L_clip-L_vf+entropy_bonus).mean()

            all_L_clip.append(L_clip.mean().item())
            all_L_vf.append(L_vf.mean().item())
            all_L_entropy.append(entropy_bonus.mean().item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        epoch_L_clip = sum(all_L_clip[-K_epochs:]) / K_epochs
        epoch_L_vf = sum(all_L_vf[-K_epochs:]) / K_epochs
        epoch_L_entropy = sum(all_L_entropy[-K_epochs:]) / K_epochs
        stats_rows.append({
            "epoch": i + 1,
            "reward": total_rewards[-1],
            "L_clip": epoch_L_clip,
            "L_vf": epoch_L_vf,
            "L_entropy": epoch_L_entropy,
        })

        if (i+1) % print_freq == 0:
            current_time = time.time()
            elapsed_time = current_time - start_time

            epoch_num = i+1
            n_recent = int(K_epochs*print_freq)

            avg_reward = sum(total_rewards[-print_freq:]) /len(total_rewards[-print_freq:])
            L_clip_mean = sum(all_L_clip[-n_recent:]) / len(all_L_clip[-n_recent:])
            L_vf_mean = sum(all_L_vf[-n_recent:]) / len(all_L_vf[-n_recent:])
            L_entr_mean = sum(all_L_entropy[-n_recent:]) / len(all_L_entropy[-n_recent:])

            train_print(epoch_num,L_clip_mean,L_vf_mean,L_entr_mean,avg_reward,elapsed_time)

    return pd.DataFrame(stats_rows)

### Hyper paramters

In [6]:
from datetime import datetime

lr = 0.0001 # Learning Rate
epsilon = 0.1 # Clipping range
gamma = 0.99 # Reward discount factor

# Loss Coeffs
c0 = 1
c1 = 0.0001
c2 = 0.01

# Train

In [7]:
Total_runs = 3
epochs = 2500
K_epochs = 5
print_freq = 100
model_dict = {}


for i in range(Total_runs):
    total_layers,layer_specs = get_random_layers()
    print("Total layers: ",total_layers)
    print(f"Layer setup: {layer_specs}")

    model = PPO_Flappy(total_layers,layer_specs)
    optimizer = torch.optim.Adam(model.parameters(),lr = lr)

    stats = Train_Network(epochs,print_freq)

    # Saving model weights
    #model_name = model_names[i]
    #torch.save(model.state_dict(),model_name)

Total layers:  3
Layer setup: [[8, 1024], [1024, 512], [512, 2048]]


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


Epoch: 100, L_clip: -1.98, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.96, Time elapsed: 3.15s
Epoch: 200, L_clip: -0.48, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.96, Time elapsed: 6.16s
Epoch: 300, L_clip: -0.27, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.86, Time elapsed: 9.14s
Epoch: 400, L_clip: -0.17, Squared loss: 0.00, Entropy loss: 0.00, Avg Rewards: -4.84, Time elapsed: 12.24s
Epoch: 500, L_clip: 0.04, Squared loss: 0.00, Entropy loss: 0.00, Avg Rewards: -4.38, Time elapsed: 15.71s
Epoch: 600, L_clip: -0.01, Squared loss: 0.00, Entropy loss: 0.00, Avg Rewards: -4.15, Time elapsed: 19.39s
Epoch: 700, L_clip: 0.01, Squared loss: 0.00, Entropy loss: 0.00, Avg Rewards: -3.63, Time elapsed: 23.60s
Epoch: 800, L_clip: 0.05, Squared loss: 0.00, Entropy loss: 0.00, Avg Rewards: -2.79, Time elapsed: 28.49s
Epoch: 900, L_clip: 0.01, Squared loss: 0.00, Entropy loss: 0.00, Avg Rewards: -0.40, Time elapsed: 35.85s
Epoch: 1000, L_clip: -0.06, Squared

libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile


Epoch: 100, L_clip: -1.63, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.88, Time elapsed: 9.69s
Epoch: 200, L_clip: -0.74, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.89, Time elapsed: 19.63s
Epoch: 300, L_clip: -0.41, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.86, Time elapsed: 29.74s
Epoch: 400, L_clip: -0.12, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.77, Time elapsed: 39.63s
Epoch: 500, L_clip: -0.05, Squared loss: 0.00, Entropy loss: 0.00, Avg Rewards: -4.81, Time elapsed: 49.85s
Epoch: 600, L_clip: -0.04, Squared loss: 0.00, Entropy loss: 0.00, Avg Rewards: -4.80, Time elapsed: 62.12s
Epoch: 700, L_clip: -0.04, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.80, Time elapsed: 72.92s
Epoch: 800, L_clip: -0.17, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.81, Time elapsed: 83.43s
Epoch: 900, L_clip: -0.12, Squared loss: 0.00, Entropy loss: 0.01, Avg Rewards: -4.87, Time elapsed: 93.93s
Epoch: 1000, L_clip: -0.07, S

In [8]:
stats

,epoch,reward,L_clip,L_vf,L_entropy
0,1,-5.0,-3.968718,0.001519,0.006820
1,2,-5.0,-4.250505,0.001827,0.006657
2,3,-5.0,-3.752351,0.001424,0.006643
3,4,-5.0,-3.613642,0.001350,0.006411
4,5,-5.0,-3.529992,0.001277,0.006634
...,...,...,...,...,...
2495,2496,675.0,0.136474,0.000021,0.003233
2496,2497,1129.0,0.061248,0.000016,0.003236
2497,2498,1637.0,0.003505,0.000013,0.003259
2498,2499,137.0,-0.171448,0.000057,0.003132
